<a href="https://colab.research.google.com/github/Joeaviator/ABOUT-JOSHUA/blob/main/Association_Rule_Base_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings("ignore")

In [17]:
df=pd.read_csv("/content/customers (1).csv")
df.head(10)

,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100),cluster
0,1,Male,19,15,39,4
1,2,Male,21,15,81,3
2,3,Female,20,16,6,4
3,4,Female,23,16,77,3
4,5,Female,31,17,40,4
5,6,Female,22,17,76,3
6,7,Female,35,18,6,4
7,8,Female,23,18,94,3
8,9,Male,64,19,3,4
9,10,Female,30,19,72,3


In [18]:
df['Age Group']=pd.cut(df['Age'],bins=[0,20,45,60,80],labels=['Children','Young Adult','Middle Aged', 'Elderly'])

In [19]:
df['Income groups']=pd.cut(df['Annual Income (k$)'],bins=[10,20,30,40],labels=['Low','Medium','High'])

In [20]:
df['Spending Group']=pd.cut(df['Spending Score (1-100)'],bins=[20,40,80,100], labels=["Low","Medium","High"])

In [21]:
df.head()

,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100),cluster,Age Group,Income groups,Spending Group
0,1,Male,19,15,39,4,Children,Low,Low
1,2,Male,21,15,81,3,Young Adult,Low,High
2,3,Female,20,16,6,4,Children,Low,NaN
3,4,Female,23,16,77,3,Young Adult,Low,Medium
4,5,Female,31,17,40,4,Young Adult,Low,Low


In [22]:
df=df.drop(columns=['CustomerID','Age','Annual Income (k$)','Spending Score (1-100)'],axis=1)

In [23]:
df.head()

,Gender,cluster,Age Group,Income groups,Spending Group
0,Male,4,Children,Low,Low
1,Male,3,Young Adult,Low,High
2,Female,4,Children,Low,NaN
3,Female,3,Young Adult,Low,Medium
4,Female,4,Young Adult,Low,Low


In [24]:
#Convert Gender into an item, base of our association
df["Gender Item"]="Gender_"+df["Gender"].astype(str)

In [25]:
#Convert Cluster into an item, base of our association
df["Cluster Item"]="Cluster_"+df["cluster"].astype(str)

In [26]:
df.isnull().sum()

,0
Gender,0
cluster,0
Age Group,0
Income groups,150
Spending Group,36
Gender Item,0
Cluster Item,0


In [27]:
df = df.dropna()

In [28]:
df.head()

,Gender,cluster,Age Group,Income groups,Spending Group,Gender Item,Cluster Item
0,Male,4,Children,Low,Low,Gender_Male,Cluster_4
1,Male,3,Young Adult,Low,High,Gender_Male,Cluster_3
3,Female,3,Young Adult,Low,Medium,Gender_Female,Cluster_3
4,Female,4,Young Adult,Low,Low,Gender_Female,Cluster_4
5,Female,3,Young Adult,Low,Medium,Gender_Female,Cluster_3


# Create customer Transactions


In [30]:
transactions=df[["Gender Item", "Cluster Item", "Income groups","Spending Group"]].astype(str).values.tolist()

In [32]:
#Display the first 5 transactions
print("First 5 transactions: ")
for i in transactions[:5]:
  print(i)

First 5 transactions: 
['Gender_Male', 'Cluster_4', 'Low', 'Low']
['Gender_Male', 'Cluster_3', 'Low', 'High']
['Gender_Female', 'Cluster_3', 'Low', 'Medium']
['Gender_Female', 'Cluster_4', 'Low', 'Low']
['Gender_Female', 'Cluster_3', 'Low', 'Medium']


In [33]:
#encode the transactions
encoder=TransactionEncoder()
#train the algorithm
encoded_data=encoder.fit(transactions).transform(transactions)

In [34]:
#Convert the encoded data into a dataframe
edf=pd.DataFrame(encoded_data,columns=encoder.columns_)
edf.head()

,Cluster_0,Cluster_3,Cluster_4,Cluster_5,Gender_Female,Gender_Male,High,Low,Medium
0,False,False,True,False,False,True,False,True,False
1,False,True,False,False,False,True,True,True,False
2,False,True,False,False,True,False,False,True,True
3,False,False,True,False,True,False,False,True,False
4,False,True,False,False,True,False,False,True,True


#Find Association


In [38]:
frequent_items=apriori(edf,min_support=0.10,use_colnames=True)
frequent_items.head()

,support,itemsets
0,0.578947,(Cluster_3)
1,0.236842,(Cluster_4)
2,0.105263,(Cluster_5)
3,0.657895,(Gender_Female)
4,0.342105,(Gender_Male)


In [39]:
#add items into the frequency items
frequent_items["length"]=frequent_items["itemsets"].apply(len)

In [40]:
#sort the data
frequent_items=frequent_items.sort_values(by="support",ascending=False)

In [41]:
#Display the addociation
frequent_items.head()

,support,itemsets,length
7,0.684211,(Medium),1
3,0.657895,(Gender_Female),1
0,0.578947,(Cluster_3),1
5,0.552632,(High),1
6,0.500000,(Low),1


In [43]:
#Apply yhe association rules
rules=association_rules(frequent_items,metric="confidence",min_threshold=0.50)


In [47]:
rules[["antecedents","consequents","support","lift","confidence"]]

,antecedents,consequents,support,lift,confidence
0,(Gender_Female),(Medium),0.473684,1.052308,0.720000
1,(Medium),(Gender_Female),0.473684,1.052308,0.692308
2,(Gender_Female),(High),0.421053,1.158095,0.640000
3,(High),(Gender_Female),0.421053,1.158095,0.761905
4,(Medium),(Cluster_3),0.421053,1.062937,0.615385
...,...,...,...,...,...
68,"(High, Medium, Cluster_5)",(Gender_Female),0.105263,1.520000,1.000000
69,"(Gender_Female, Cluster_5)","(Medium, High)",0.105263,3.454545,1.000000
70,"(Medium, Cluster_5)","(Gender_Female, High)",0.105263,2.375000,1.000000
71,"(High, Cluster_5)","(Gender_Female, Medium)",0.105263,2.111111,1.000000
